# A Movie Recommendation Service
### Source: https://www.codementor.io/@jadianes/building-a-recommender-with-apache-spark-python-example-app-part1-du1083qbw

#### Create a SparkContext configured for local mode



In [2]:
import pyspark
sc = pyspark.SparkContext('local[*]')

#### File download
Small: 100,000 ratings and 3,600 tag applications applied to 9,000 movies by 600 users. Last updated 9/2018.
Full: approximately 33,000,000 ratings and 2,000,000 tag applications applied to 86,000 movies by 330,975 users. 



In [3]:
complete_dataset_url = 'http://files.grouplens.org/datasets/movielens/ml-latest.zip'


#### Download location(s)


In [4]:
import os
datasets_path = os.path.join('/home/jovyan', 'work')
complete_dataset_path = os.path.join(datasets_path,'ml-latest.zip')


#### Getting file(s)


In [5]:
import urllib.request
complete_f = urllib.request.urlretrieve (complete_dataset_url, complete_dataset_path)

#### Extracting file(s)


In [5]:
import zipfile
    
with zipfile.ZipFile(complete_dataset_path, "r") as z:
    z.extractall(datasets_path)

## Loading and parsing datasets
Now we are ready to read in each of the files and create an RDD consisting of parsed lines. 

Each line in the ratings dataset (ratings.csv) is formatted as: 
+ userId,movieId,rating,timestamp 

Each line in the movies (movies.csv) dataset is formatted as:
+ movieId,title,genres 

The format of these files is uniform and simple, so we can use Python split() to parse their lines once they are loaded into RDDs. Parsing the movies and ratings files yields two RDDs: 
+ For each line in the ratings dataset, we create a tuple of (UserID, MovieID, Rating). We drop the timestamp because we do not need it for this recommender.
+ For each line in the movies dataset, we create a tuple of (MovieID, Title). We drop the genres because we do not use them for this recommender.

#### ratings.csv


In [6]:
complete_ratings_file = os.path.join(datasets_path, 'ml-latest', 'ratings.csv')
complete_ratings_raw_data = sc.textFile(complete_ratings_file)
complete_ratings_raw_data_header = complete_ratings_raw_data.take(1)[0]
# Parse
complete_ratings_data = complete_ratings_raw_data.filter(lambda line: line!=complete_ratings_raw_data_header)\
    .map(lambda line: line.split(",")).map(lambda tokens: (int(tokens[0]),int(tokens[1]),float(tokens[2]))).cache()

print ('There are {} recommendations in the complete dataset'.format(complete_ratings_data.count()))
complete_ratings_data.take(3)

There are 33832162 recommendations in the complete dataset


[(1, 1, 4.0), (1, 110, 4.0), (1, 158, 4.0)]

#### movies.csv


In [7]:
complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

complete_movies_data = complete_movies_raw_data.filter(lambda line: line != complete_movies_raw_data_header)\
    .map(lambda line: line.split(","))\
    .map(lambda tokens: (int(tokens[0]), tokens[1], tokens[2])).cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]), x[1]))
print('There are {} movies in the complete dataset'.format(complete_movies_titles.count()))
complete_movies_data.take(3)



There are 86537 movies in the complete dataset


[(1, 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy'),
 (2, 'Jumanji (1995)', 'Adventure|Children|Fantasy'),
 (3, 'Grumpier Old Men (1995)', 'Comedy|Romance')]

## Collaborative Filtering
In Collaborative filtering we make predictions (filtering) about the interests of a user by collecting preferences or taste information from many users (collaborating). The underlying assumption is that if a user A has the same opinion as a user B on an issue, A is more likely to have B's opinion on a different issue x than to have the opinion on x of a user chosen randomly. 

At first, people rate different items (like videos, images, games). Then, the system makes predictions about a user's rating for an item not rated yet. The new predictions are built upon the existing ratings of other users with similar ratings with the active user. In the image, the system predicts that the user will not like the video.

Spark MLlib library for Machine Learning provides a Collaborative Filtering implementation by using Alternating Least Squares. The implementation in MLlib has the following parameters:

+ numBlocks is the number of blocks used to parallelize computation (set to -1 to auto-configure).
+ rank is the number of latent factors in the model.
+ iterations is the number of iterations to run.
+ lambda specifies the regularization parameter in ALS.
+ implicitPrefs specifies whether to use the explicit feedback ALS variant or one adapted for implicit feedback data.
+ alpha is a parameter applicable to the implicit feedback variant of ALS that governs the baseline confidence in preference observations.

#### Selecting ALS parameters using the small dataset
In order to determine the best ALS parameters, we will use the small dataset. We need first to split it into train, validation, and test datasets.

In [8]:
sampled_data = complete_ratings_data.sample(False, 0.5, seed=42)

training_RDD, validation_RDD, test_RDD = sampled_data.randomSplit([6, 2, 2], seed=0)
validation_for_predict_RDD = validation_RDD.map(lambda x: (x[0], x[1]))
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))

from pyspark.mllib.recommendation import ALS
import math

seed = 5
iterations = 10
regularization_parameter = 0.1
ranks = [4, 8, 12]
errors = [0, 0, 0]
err = 0
tolerance = 0.02

min_error = float('inf')
best_rank = -1

for rank in ranks:
    model = ALS.train(training_RDD, rank, seed=seed, iterations=iterations,
                      lambda_=regularization_parameter)
    predictions = model.predictAll(validation_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
    rates_and_preds = validation_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
    error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())
    errors[err] = error
    err += 1
    print(f'For rank {rank} the RMSE is {error}')
    if error < min_error:
        min_error = error
        best_rank = rank

print(f'The best model was trained with rank {best_rank}')

For rank 4 the RMSE is 0.8436082865473064
For rank 8 the RMSE is 0.8320920924141935
For rank 12 the RMSE is 0.8303393140413126
The best model was trained with rank 12


#### Training phase


In [9]:
model = ALS.train(training_RDD, best_rank, seed=seed, iterations=iterations, lambda_=regularization_parameter)
predictions = model.predictAll(test_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
rates_and_preds = test_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())
print('For testing data the RMSE is %s' % (error))

For testing data the RMSE is 0.8304077099594457


## How to make recommendations
Although we aim at building an online movie recommender, now that we know how to have our recommender model ready, we can give it a try providing some movie recommendations. This will help us coding the recommending engine later on when building the web service, and will explain how to use the model in any other circumstances.

When using collaborative filtering, getting recommendations is not as simple as predicting for the new entries using a previously generated model. Instead, we need to train again the model but including the new user preferences in order to compare them with other users in the dataset. That is, the recommender needs to be trained every time we have new user ratings (although a single model can be used by multiple users of course!). This makes the process expensive, and it is one of the reasons why scalability is a problem (and Spark a solution!). Once we have our model trained, we can reuse it to obtain top recomendations for a given user or an individual rating for a particular movie. These are less costly operations than training the model itself.

Another thing we want to do, is give recommendations of movies with a certain minimum number of ratings. For that, we need to count the number of ratings per movie.

In [10]:
complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

complete_movies_data = complete_movies_raw_data.filter(lambda line: line!=complete_movies_raw_data_header)\
    .map(lambda line: line.split(",")).map(lambda tokens: (int(tokens[0]),tokens[1],tokens[2])).cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]),x[1]))
    
print ("There are %s movies in the complete dataset" % (complete_movies_titles.count()))

There are 86537 movies in the complete dataset


In [12]:
def get_counts_and_averages(ID_and_ratings_tuple):
    nratings = len(ID_and_ratings_tuple[1])
    return ID_and_ratings_tuple[0], (nratings, float(sum(x for x in ID_and_ratings_tuple[1]))/nratings)

movie_ID_with_ratings_RDD = (complete_ratings_data.map(lambda x: (x[1], x[2])).groupByKey())
movie_ID_with_avg_ratings_RDD = movie_ID_with_ratings_RDD.map(get_counts_and_averages)
movie_rating_counts_RDD = movie_ID_with_avg_ratings_RDD.map(lambda x: (x[0], x[1][0]))

### Adding new user ratings
Now we need to rate some movies for the new user. We will put them in a new RDD and we will use the user ID 0, that is not assigned in the MovieLens dataset. Check the dataset movies file for ID to Tittle assignment (so you know what movies are you actually rating).

In [13]:
# User 1 ratings (userId = 0)
new_user_1_ID = 0
new_user_1_ratings = [
    (0, 260, 4),   # Star Wars (1977)
    (0, 1, 3),     # Toy Story (1995)
    (0, 16, 3),    # Casino (1995)
    (0, 25, 4),    # Leaving Las Vegas (1995)
    (0, 32, 4),    # Twelve Monkeys (1995)
    (0, 335, 1),   # The Flintstones (1994)
    (0, 379, 1),   # Timecop (1994)
    (0, 296, 3),   # Pulp Fiction (1994)
    (0, 858, 5),   # The Godfather (1972)
    (0, 50, 4)     # The Usual Suspects (1995)
]
new_user_1_ratings_RDD = sc.parallelize(new_user_1_ratings)

# User 2 ratings (userId = 999999)
new_user_2_ID = 999999
new_user_2_ratings = [
    (999999, 110, 4),    # Braveheart (1995)
    (999999, 47, 5),     # Seven (1995)
    (999999, 150, 3),    # Apollo 13 (1995)
    (999999, 527, 4),    # Schindler's List (1993)
    (999999, 589, 2),    # Terminator 2 (1991)
    (999999, 1196, 4),   # Empire Strikes Back (1980)
    (999999, 1210, 5),   # Return of the Jedi (1983)
    (999999, 1240, 4),   # Terminator (1984)
    (999999, 1291, 5),   # Last Crusade (1989)
    (999999, 2571, 4)    # The Matrix (1999)
]
new_user_2_ratings_RDD = sc.parallelize(new_user_2_ratings)


Now we add them to the data we will use to train our recommender model. We use Spark's union() transformation for this.

In [22]:
all_new_users_ratings_RDD = new_user_1_ratings_RDD.union(new_user_2_ratings_RDD)

complete_data_with_new_ratings_RDD = complete_ratings_data.union(all_new_users_ratings_RDD).cache()

print("Total number of ratings including new users:", complete_data_with_new_ratings_RDD.count())


Total number of ratings including new users: 33832182


And finally we train the ALS model using all the parameters we selected before (when using the small dataset).

In [24]:
from time import time

t0 = time()
new_ratings_model = ALS.train(complete_data_with_new_ratings_RDD, best_rank, seed=seed, 
                              iterations=iterations, lambda_=regularization_parameter)
tt = time() - t0

print ("New model trained in %s seconds" % round(tt,3))

New model trained in 256.15 seconds


## Getting top recommendations
Let's now get some recommendations! For that we will get an RDD with all the movies the new user hasn't rated yet. We will them together with the model to predict ratings.

In [25]:
def get_top_recommendations_for_user(user_id, ratings_model, min_ratings=25, top_n=15):
    # Get movies already rated by the user
    user_rated_movie_ids = complete_data_with_new_ratings_RDD \
        .filter(lambda x: x[0] == user_id) \
        .map(lambda x: x[1]) \
        .collect()

    # Get movies not yet rated
    unrated_movies_RDD = complete_movies_data \
        .filter(lambda x: x[0] not in user_rated_movie_ids) \
        .map(lambda x: (user_id, x[0]))

    # Predict ratings
    predictions_RDD = ratings_model.predictAll(unrated_movies_RDD)
    predicted_ratings_RDD = predictions_RDD.map(lambda x: (x.product, x.rating))

    # Join with movie titles and rating counts
    predicted_with_titles_counts_RDD = predicted_ratings_RDD \
        .join(complete_movies_titles) \
        .join(movie_rating_counts_RDD) \
        .map(lambda r: (r[1][0][1], r[1][0][0], r[1][1]))  # (title, predicted_rating, count)

    # Filter and get top recommendations
    top_movies = predicted_with_titles_counts_RDD \
        .filter(lambda r: r[2] >= min_ratings) \
        .takeOrdered(top_n, key=lambda x: -x[1])

    print('TOP recommended movies (with more than %d reviews):\n%s' %
          (min_ratings, '\n'.join(map(str, top_movies))))


In [26]:
# User 1 (ID = 0), Scenario 1 (min 25 ratings)
get_top_recommendations_for_user(0, new_ratings_model, min_ratings=25)

# User 1, Scenario 2 (min 100 ratings)
get_top_recommendations_for_user(0, new_ratings_model, min_ratings=100)

# User 2 (ID = 999999), Scenario 1
get_top_recommendations_for_user(999999, new_ratings_model, min_ratings=25)

# User 2, Scenario 2
get_top_recommendations_for_user(999999, new_ratings_model, min_ratings=100)


TOP recommended movies (with more than 25 reviews):
("Long Night's Journey Into Day (2000)", 3.9646281871172286, 36)
('Mugabe and the White African (2009)', 3.9498446946255026, 27)
("Sharpe's Rifles (1993)", 3.8309537740592594, 28)
('"Godfather: Part II', 3.827675822307538, 47271)
('The Cube (1969)', 3.782515187851595, 25)
('"Last Lions', 3.778484510894639, 51)
('"Come Sweet Death (Komm', 3.7718309043605323, 31)
('"Lewis Black: Red', 3.769084164479228, 25)
("Schindler's List (1993)", 3.753015553908138, 84232)
('"Human Condition III', 3.744716126782618, 145)
('Interrogation (Przesluchanie) (1989)', 3.7223063604249846, 40)
('Norm MacDonald: Me Doing Standup (2011)', 3.7213856361766946, 36)
('"Shawshank Redemption', 3.7205372966690526, 122296)
('As I Was Moving Ahead Occasionally I Saw Brief Glimpses of Beauty (2000)', 3.7127667201124073, 26)
('8:46 (2020)', 3.6993027544402435, 38)
TOP recommended movies (with more than 100 reviews):
('"Godfather: Part II', 3.827675822307538, 47271)
("Sch

## Interpretation and Insights on Recommendation Scenarios

This section presents an analysis of movie recommendation results for two users under two distinct filtering scenarios. The primary goal is to compare how the recommendation model behaves when it includes movies with at least 25 ratings versus those with at least 100 ratings. These two thresholds allow us to assess the trade-off between personalization and recommendation reliability.

### User 1 (User ID: 0)

**Scenario 1 (Minimum 25 ratings):**  
The recommendations for User 1 in this scenario feature several lesser-known titles such as *Long Night's Journey Into Day*, *Mugabe and the White African*, and *Come Sweet Death*. These are niche documentaries or foreign films that received high ratings from a small group of users. The results reflect a strong degree of personalization, potentially surfacing content that closely aligns with the user's unique preferences. However, due to the lower number of total ratings, the reliability of these suggestions can be less robust, as they are based on a smaller sample size.

**Scenario 2 (Minimum 100 ratings):**  
In contrast, this scenario presents widely recognized films like *The Godfather: Part II*, *Schindler's List*, and *The Shawshank Redemption*. These are classic titles with strong consensus among a large number of users. The results are highly reliable and safe, offering well-established content that is broadly liked. However, personalization is less apparent here since the model gravitates toward universally acclaimed titles, reducing the diversity and uniqueness of the recommendations.

### User 2 (User ID: 999999)

**Scenario 1 (Minimum 25 ratings):**  
The recommendations in this case include *Mushishi: The Shadow That Devours the Sun*, *Cat City*, and *Cosmos*. These titles suggest that the user’s profile aligns with viewers who prefer documentaries, anime, or science content. The inclusion of less mainstream films demonstrates the collaborative filtering model's ability to capture unique preferences. However, like User 1's Scenario 1, this approach is more prone to overfitting or noise due to the smaller number of ratings used as a basis.

**Scenario 2 (Minimum 100 ratings):**  
User 2’s recommendations here shift back to highly rated and widely viewed titles such as *The Lord of the Rings* trilogy, *Fight Club*, and *Amélie*. These results highlight popular and critically acclaimed films, providing a high degree of confidence in the model’s predictions. The trade-off is a loss in nuanced personalization. While the content is likely to be of high quality, it may not fully reflect the user's distinct interests.

### Overall Analysis

Scenario 1 excels in personalization, offering users a chance to discover unique, lesser-known movies that align with their niche interests. However, it comes with the caveat of lower statistical confidence, since movies with fewer ratings can be disproportionately influenced by a small number of opinions. Scenario 2 provides recommendations that are reliable and broadly appealing, benefiting from the stability of a large sample size, but it may lack variety or originality in the suggestions.

A key takeaway is that both scenarios have clear advantages. A well-rounded recommendation system could benefit from incorporating both approaches—using lower thresholds to discover unique content and higher thresholds to ensure quality and reliability. This hybrid strategy could deliver balanced recommendations that are both personalized and trustworthy.
